In [21]:
import os, re
import tiktoken
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
BOOK_NAMES = {
    "BPHS_English_Book.md":   "Brihat Parashara Hora Shastra (English)",
    "BPHS_Hindi_Book.md":     "Brihat Parashara Hora Shastra (Hindi)",
    "Phaladeepika_part_1.md": "Phaladeepika Part 1",
    "Phaladeepika_part_2.md": "Phaladeepika Part 2",
    "Saravali.md":            "Saravali",
}
TOC_END_PATTERNS = [
    r'\n# (?:Preface|Chapter \d)',
    r'\nSARAVALI\n# Chapter',
    r'\n# PHALADEEPIKA\b',
]

_enc = tiktoken.get_encoding("cl100k_base")

In [ ]:
def remove_toc(text: str) -> str:
    for pattern in TOC_END_PATTERNS:
        match = re.search(pattern, text)
        if match:
            return text[match.start():]
    return text

def token_len(text: str) -> int:  #Return length of the tokens--
    return len(_enc.encode(text))

In [ ]:
import re

def split_by_chapters(text):
    # These books use "Chapter N" or "Chapter N: Title" as real boundaries
    pattern = r'(?=# Chapter \d+)'
    chunks = re.split(pattern, text)
    return [c.strip() for c in chunks if len(c.strip()) > 200]

In [ ]:
char_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    separators=["\n**Sloka", "\n**Verse", "\n**Stanza", "\n\n", "।", ". ", "? ", "! ", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=75,
    keep_separator=True,
)


In [ ]:
def split_documents(docs):
    final_chunks = []
    for doc in docs:
        filename = os.path.basename(doc.metadata.get("source", ""))
        book_name = BOOK_NAMES.get(filename, filename)

        # Normalize line endings + strip ToC
        clean_text = doc.page_content.replace('\r\n', '\n').replace('\r', '\n')
        clean_text = remove_toc(clean_text)

        md_chunks = header_splitter.split_text(clean_text)

        for chunk in md_chunks:
            content = chunk.page_content.strip()
            if token_len(content) < 30:   # skip micro-chunks
                continue

            chunk.metadata["book"] = book_name
            chunk.metadata["source"] = doc.metadata.get("source", "")
            # drop the noisy h1 key entirely
            chunk.metadata.pop("h1", None)

            if token_len(content) > 400:
                sub_chunks = char_splitter.split_documents([chunk])
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk)

    print(f'Total chunks created: {len(final_chunks)}')
    return final_chunks

5

In [27]:
pip install langchain-core


Note: you may need to restart the kernel to use updated packages.


In [5]:
def split_documents(docs):
    final_chunks = []

    for doc in docs:
        # Stage 1: header-based split
        md_chunks = header_splitter.split_text(doc.page_content)

        for chunk in md_chunks:
            # carry over source filename into every chunk's metadata
            chunk.metadata.update(doc.metadata)

            # Stage 2: only re-split if chunk is too large
            if len(chunk.page_content) > 700:
                sub_chunks = char_splitter.split_documents([chunk])
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk)

    print(f'Total chunks created: {len(final_chunks)}')
    return final_chunks


if __name__ == "__main__":
    from document_loader import load_documents
    docs = load_documents()
    chunks = split_documents(docs)

    # inspect a sample
    print("\n--- Sample Chunk ---")
    print(chunks[50].page_content)
    print("\n--- Metadata ---")
    print(chunks[10].metadata)


Document 5 Loaded
Total chunks created: 7305

--- Sample Chunk ---
# 37. LANARYOGAS  
Adhiyoga from the Moon, Dhana, Sunapha, Duradhura yogas, Ar.apba, and Kemadruma.

--- Metadata ---
{'book': 'ANTIDOTES FOR', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}


In [30]:
import re
import os

BOOK_NAMES = {
    "BPHS_English_Book.md":   "Brihat Parashara Hora Shastra (English)",
    "BPHS_Hindi_Book.md":     "Brihat Parashara Hora Shastra (Hindi)",
    "Phaladeepika_part_1.md": "Phaladeepika Part 1",
    "Phaladeepika_part_2.md": "Phaladeepika Part 2",
    "Saravali.md":            "Saravali",
}

def split_by_chapters(text):
    pattern = r'(?=# Chapter \d+)'
    chunks = re.split(pattern, text)
    return [c.strip() for c in chunks if len(c.strip()) > 200]

def split_documents(docs):
    final_chunks = []

    for doc in docs:
        filename = os.path.basename(doc.metadata.get("source", ""))
        book_name = BOOK_NAMES.get(filename, filename)

        # normalize line endings
        clean_text = doc.page_content.replace('\r\n', '\n').replace('\r', '\n')

        # stage 1: split by chapter
        chapter_chunks = split_by_chapters(clean_text)

        for chapter_text in chapter_chunks:
            # extract chapter title from first line
            first_line = chapter_text.split('\n')[0].strip()
            chapter_title = first_line.replace('#', '').strip()

            from langchain_core.documents import Document
            chunk_doc = Document(
                page_content=chapter_text,
                metadata={
                    "book": book_name,
                    "chapter": chapter_title,
                    "source": doc.metadata.get("source", "")
                }
            )

            # stage 2: re-split if too large
            if len(chapter_text) > 700:
                sub_chunks = char_splitter.split_documents([chunk_doc])
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk_doc)

    print(f'Total chunks created: {len(final_chunks)}')
    return final_chunks

In [31]:
# test it quickly
chunks = split_documents(docs)
print(f"Total: {len(chunks)}")

# check first 20 chunks
for i, c in enumerate(chunks[:20]):
    print(f"\n[{i}] {c.metadata}")
    print(c.page_content[:150])

Total chunks created: 5356
Total: 5356

[0] {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'BrihatParasara Hora Sastra', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
# BrihatParasara Hora Sastra

# of MAIIARSHI PARASARA

# vol, I

English translation, commentary, annotation and editing by R. SANTHANAM

Ranjan Publi

[1] {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'BrihatParasara Hora Sastra', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
| Preface                                | translator by          | 10                                    | |
| --------------------------------------

[2] {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'BrihatParasara Hora Sastra', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
. SPECIAL ASCENDANTS                  |                        |                                       | |
| Bhava Lagna,                           | 

[3] {'boo